# Phase 1 — Akash: Corpus Acquisition, Extraction, Segmentation

**RBI-ObliBench / Agentic RAG compliance system**

Runs the full corpus pipeline at scale: discovery -> download -> extraction ->
segmentation -> cross-reference resolution against the live RBI Master
Directions listing. Thin wrapper only — every cell calls into `src/` or
`scripts/run_harvest.py`; no pipeline logic lives here.

Requires **Internet: On** in notebook settings.

## 1. Get the code

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/karanLokhande29/Capstone_project.git"
BRANCH = "main"

WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
REPO_DIR = os.path.join(WORKING, "Capstone_project")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # /kaggle/working commonly survives across "Run All" within the same
    # Kaggle session, so a directory left over from an earlier run must not
    # be silently reused — always sync it to the latest commit on BRANCH
    # rather than trusting whatever was checked out last time.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True,
).stdout.strip()
print("repository:", REPO_DIR)
print("commit:     ", commit, "-- check this matches the latest commit on GitHub before trusting a run")

## 2. Dependencies

`pdfplumber` is the one dependency this branch needs beyond Kaggle's base
image (see `requirements.txt` for what's already present).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pdfplumber"], check=True)

from src.common.verify import environment_versions
for name, version in environment_versions().items():
    print(f"{name:12s} {version}")

## 3. Unit + integration tests

No real network calls in these — discovery/download/extraction/segmentation
are all tested against fixtures. This confirms the code is sound before
spending the full harvest's network time.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "tests/test_scraper_rbi.py", "tests/test_extraction_text.py",
     "tests/test_preprocessing_segmenter.py", "tests/test_preprocessing_integration.py",
     "-q"],
    capture_output=True, text=True,
)
print(result.stdout[-4000:])
if result.returncode != 0:
    print(result.stderr[-4000:])
print("TESTS:", "PASS" if result.returncode == 0 else "FAIL")

## 4. Small validation slice

A quick, small-scale run first — cheap to re-run if something's wrong before
committing to the full ~380-document harvest below.

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/run_harvest.py", "all", "--limit", "12", "--json"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-4000:])

## 5. Full corpus harvest (Week 3)

Discovers and downloads the entire Master Directions listing — count is
**discovered, not assumed**; whatever the real listing returns is what gets
reported. Rate-limited per `config.yaml`'s `network.rate_limit_sec`, so this
takes a while (budget roughly 15-20 minutes for ~380 documents).

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "scripts/run_harvest.py", "all", "--json"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-4000:])

## 6. Result

Copy into the weekly logbook.

The report is read back through `PathResolver` rather than a hardcoded
relative path — `config.yaml`'s Kaggle working root is `/kaggle/working`,
one level *above* where this notebook cloned the repo (`/kaggle/working/
Capstone_project`), so everything `scripts/run_harvest.py` writes lands
outside the cloned directory. A plain `open("reports/...")` here would look
in the wrong place; resolving the key is what actually finds it in either
environment.

In [ ]:
from src.common.config import load_config
from src.common.paths import PathResolver

cfg = load_config()
resolver = PathResolver.from_config(cfg)
report_path = resolver.read_path("reports", "phase1_akash_corpus.md")
print(f"reading: {report_path}\n")
print(report_path.read_text())

---

### Saving the corpus as a Kaggle Dataset

1. Confirm `data/raw/`, `data/extracted/`, `data/processed/`, `data/cache/`,
   and `data/metadata/` are populated under `/kaggle/working/`.
2. **New Dataset** (or **New Version** on an existing one) named e.g.
   `rbi-corpus-v1`, including those five directories.
3. Add the dataset slug to `environment.kaggle.input_datasets` in
   `config/config.yaml` and commit it, so the next session — and Karan's and
   Meer's notebooks — read this corpus as read-only input via **Add Input**
   instead of re-harvesting it.
4. Download at least the small-slice output back to your local machine before
   starting the next branch's work.